## Import

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"   # single-threaded so parallel jobs do not fight over cores

import json, pickle, time
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## Setup

In [2]:
SEED = 42
np.random.seed(SEED)
K, REPEATS = 5, 1                      # scratch twin: one repeat only (repeat 1 = the same folds as the pytorch twin)
KP = "weights_V4_1/regression_pytorch_kfold_metrics.json"
assert os.path.exists(KP), "pytorch K-Fold twin must run first (epochs come from it)"
ORA = {(r["model"], r["repeat"], r["fold"]): r for r in json.load(open(KP))["folds"]}
os.makedirs("weights_V4_1", exist_ok=True)
DELTA, NF = 0.5, 13
VARS = {"ReLU (baseline)": dict(ffn="relu", hid=64), "ReGLU, param-matched": dict(ffn="reglu", hid=38)}
NAME = "NAM (log1p + Huber) | "

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims | NAM delta={DELTA} | log1p target, Huber loss (robust to right-skewed income)")

features: 13 dims | NAM delta=0.5 | log1p target, Huber loss (robust to right-skewed income)


## Dataloader

In [4]:
def prep_split(tr_idx, te_idx):
    """Standardise using the training rows only (no missing values left in the regression set)."""
    sc = StandardScaler().fit(X_full[tr_idx])
    return sc.transform(X_full[tr_idx]), sc.transform(X_full[te_idx]), y_raw[tr_idx], y_raw[te_idx]

## Model

In [5]:
def init_params(ffn, hid):
    """ReLU: W1 is (F, 1, hid). ReGLU: W1 is (F, 1, 2*hid) = [value | gate]."""
    global FFN, HH, HW, P_
    FFN, HH = ffn, hid; HW = hid if ffn == "relu" else 2 * hid
    P_ = {}
    P_["W1"] = np.random.randn(NF, 1, HW) * np.sqrt(2.0 / 1.0); P_["b1"] = np.zeros((NF, HW))
    P_["W2"] = np.random.randn(NF, HH, 1) * np.sqrt(2.0 / HH); P_["b2"] = np.zeros((NF, 1))
    P_["b0"] = np.zeros(1)
    return P_
for v, s in VARS.items():
    print(f"{v:22s} scratch parameters: {sum(a.size for a in init_params(**s).values())}")

ReLU (baseline)        scratch parameters: 2510
ReGLU, param-matched   scratch parameters: 2484


In [6]:
def act(Z1):
    if FFN == "reglu": return Z1[..., :HH] * np.maximum(Z1[..., HH:], 0)      # value * ReLU(gate)
    return np.maximum(Z1, 0)
def fwd(Xb):
    B = Xb.shape[0]
    Z1 = Xb[:, :, None] * P_["W1"][:, 0, :][None, :, :] + P_["b1"][None, :, :]  # (B,F,HW)
    A1 = act(Z1)
    Fj = np.einsum("bfh,fh->bf", A1, P_["W2"][:, :, 0]) + P_["b2"].ravel()  # (B,F)
    pred = Fj.sum(1, keepdims=True) + P_["b0"]
    return pred, (Xb, Z1, A1, Fj)

In [7]:
def huber(pred, y):
    e = (pred.ravel() - y.ravel()); B = len(e)
    q = np.abs(e) <= DELTA  # tie -> quadratic branch (torch parity)
    loss = np.where(q, 0.5 * e ** 2, DELTA * (np.abs(e) - 0.5 * DELTA)).mean()
    dp = np.where(q, e, DELTA * np.sign(e)).reshape(-1, 1) / B
    return float(loss), dp

In [8]:
def bwd(dp, cache, G):
    Xb, Z1, A1, Fj = cache; B = Xb.shape[0]
    G["b0"] += dp.sum(0)
    dF = np.broadcast_to(dp, (B, NF))                       # dF[b,f]
    G["b2"] += dF.sum(0).reshape(NF, 1)
    G["W2"] += np.einsum("bfh,bf->fh", A1, dF)[:, :, None]  # dW2[f,h,0] = sum_b A1*dF
    dA = dF[:, :, None] * P_["W2"][None, :, :, 0]            # dA[b,f,h] = dF*W2
    if FFN == "reglu":
        val, gate = Z1[..., :HH], Z1[..., HH:]
        dZ = np.concatenate([dA * np.maximum(gate, 0), dA * val * (gate > 0)], axis=-1)
    else:
        dZ = dA * (Z1 > 0)
    G["b1"] += dZ.sum(0)
    G["W1"] += np.einsum("bf,bfh->fh", Xb, dZ)[:, None, :]

## Sanity Check

In [9]:
init_params(**VARS["ReGLU, param-matched"])
rs = np.random.RandomState(1); Xc, yc = rs.randn(2, NF), rs.randn(2)
p0, c0 = fwd(Xc); _, dp0 = huber(p0, yc)
G0 = {k: np.zeros_like(a) for k, a in P_.items()}; bwd(dp0, c0, G0)
for key, idx in [("W2", (0, 0, 0)), ("W1", (1, 0, 3)), ("W1", (1, 0, HH + 3)), ("b1", (4, HH + 2)), ("b2", (2, 0))]:
    e = 1e-5; old = P_[key][idx]
    P_[key][idx] = old + e; lp = huber(fwd(Xc)[0], yc)[0]
    P_[key][idx] = old - e; lm = huber(fwd(Xc)[0], yc)[0]
    P_[key][idx] = old
    num, ana = (lp - lm) / (2 * e), G0[key][idx]
    err = abs(num - ana) / (abs(num) + abs(ana) + 1e-12)
    print(f"check {key}{list(idx)}: numeric {num:.6e} analytic {ana:.6e} | relerr {err:.2e}")
    assert err < 1e-4, "backward pass does not match the numeric slope"

check W2[0, 0, 0]: numeric 4.071595e-02 analytic 4.071595e-02 | relerr 3.32e-11
check W1[1, 0, 3]: numeric 6.009858e-02 analytic 6.009858e-02 | relerr 4.59e-11
check W1[1, 0, 41]: numeric -5.062412e-02 analytic -5.062412e-02 | relerr 7.62e-11
check b1[4, 40]: numeric 1.371723e-02 analytic 1.371723e-02 | relerr 1.78e-10
check b2[2, 0]: numeric 5.000000e-01 analytic 5.000000e-01 | relerr 7.83e-12


## Training

In [10]:
LR0, WD, B1, B2, EP = 1e-3, 1e-4, 0.9, 0.999, 1e-8
def train_scratch(Xa, za, spec, E, seed):
    """Adam + cosine for E epochs (E chosen by the pytorch twin's inner validation on this fold). Leaves the trained weights in P_."""
    np.random.seed(seed); init_params(**spec)
    M = {k: np.zeros_like(a) for k, a in P_.items()}; V = {k: np.zeros_like(a) for k, a in P_.items()}
    N = len(Xa); step = 0
    for epoch in range(1, E + 1):
        lr = 0.5 * LR0 * (1 + np.cos(np.pi * (epoch - 1) / 100))
        perm = np.random.permutation(N)
        for i in range(0, N, 64):
            idx = perm[i:i+64]
            pred, cache = fwd(Xa[idx]); _, dp = huber(pred, za[idx])
            G = {k: np.zeros_like(a) for k, a in P_.items()}
            bwd(dp, cache, G); step += 1
            for k in P_:
                g = G[k] + WD * P_[k]
                M[k] = B1 * M[k] + (1 - B1) * g; V[k] = B2 * V[k] + (1 - B2) * (g ** 2)
                mh = M[k] / (1 - B1 ** step); vh = V[k] / (1 - B2 ** step)
                P_[k] -= lr * (mh / (np.sqrt(vh) + EP))

state, last = {"rows": [], "scores": {}}, {}
for fold, (a, b) in enumerate(KFold(K, shuffle=True, random_state=SEED).split(X_full), start=1):
    Xa, Xb, ya, yb = prep_split(a, b); Xa, Xb = Xa.astype(np.float64), Xb.astype(np.float64)
    ysc = StandardScaler().fit(np.log1p(ya).reshape(-1, 1)); MU, SIG = float(ysc.mean_[0]), float(ysc.scale_[0])
    za = ysc.transform(np.log1p(ya).reshape(-1, 1)).ravel()
    for vname, spec in VARS.items():
        t0 = time.time(); o_ = ORA[(NAME + vname, 1, fold)]
        train_scratch(Xa, za, spec, int(o_["epochs"]), SEED + 7 + fold)
        d = np.expm1(np.clip(fwd(Xb)[0].ravel() * SIG + MU, 0.0, 15.0)); mse = float(mean_squared_error(yb, d))
        state["rows"].append(dict(model=NAME + vname, tier="extracurricular", selection="inner validation", repeat=1, fold=fold, n_test=int(len(yb)),
            r2=round(float(r2_score(yb, d)), 4), rmse=round(float(np.sqrt(mse)), 4), mae=round(float(mean_absolute_error(yb, d)), 4), mse=round(mse, 4),
            epochs=int(o_["epochs"]), n_params=int(sum(x.size for x in P_.values())), seconds=round(time.time() - t0, 1)))
        state["scores"].setdefault(f"{NAME + vname}|r1", np.full(len(y_raw), np.nan))[b] = d
        last[vname] = dict(spec=spec, P={k: x.copy() for k, x in P_.items()}, Xb=Xb, zb=ysc.transform(np.log1p(yb).reshape(-1, 1)).ravel())
        print(f"fold {fold} {vname:22s} R2 {state['rows'][-1]['r2']:.4f} (pytorch {o_['r2']:.4f}) | {state['rows'][-1]['seconds']:.0f}s", flush=True)

fold 1 ReLU (baseline)        R2 0.7107 (pytorch 0.7219) | 2s


fold 1 ReGLU, param-matched   R2 0.7608 (pytorch 0.7492) | 3s


fold 2 ReLU (baseline)        R2 0.7780 (pytorch 0.7809) | 2s


fold 2 ReGLU, param-matched   R2 0.6558 (pytorch 0.6611) | 4s


fold 3 ReLU (baseline)        R2 0.7699 (pytorch 0.7694) | 2s


fold 3 ReGLU, param-matched   R2 0.7895 (pytorch 0.7888) | 4s


fold 4 ReLU (baseline)        R2 0.7072 (pytorch 0.7120) | 2s


fold 4 ReGLU, param-matched   R2 0.7229 (pytorch 0.6645) | 1s


fold 5 ReLU (baseline)        R2 0.6977 (pytorch 0.7024) | 3s


fold 5 ReGLU, param-matched   R2 0.7337 (pytorch 0.7770) | 3s


## K-Fold Summary

In [11]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['r2', 'rmse', 'mae']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g} for m, g in kdf.groupby("model")}
print(kdf.groupby("model")["r2"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4).to_string())
pt = pd.DataFrame([r for r in ORA.values() if r["repeat"] == 1 and r["model"] in {NAME + v for v in VARS}])
print("pytorch twin, same repeat (mean R2):", pt.groupby("model")["r2"].mean().round(4).to_dict())
json.dump({"k": K, "repeats": REPEATS, "folds": state["rows"], "summary": summary}, open("weights_V4_1/regression_scratch_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V4_1/regression_scratch_kfold_scores.npz", **state["scores"])

                                              mean     std
model                                                     
NAM (log1p + Huber) | ReLU (baseline)       0.7327  0.0381
NAM (log1p + Huber) | ReGLU, param-matched  0.7325  0.0501
pytorch twin, same repeat (mean R2): {'NAM (log1p + Huber) | ReGLU, param-matched': 0.7281, 'NAM (log1p + Huber) | ReLU (baseline)': 0.7373}


## Parity Check

In [12]:
import warnings
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             log_loss, average_precision_score, mean_squared_error, mean_absolute_error, r2_score)
warnings.filterwarnings("ignore")
torch.set_num_threads(1)

device = torch.device("cpu")


class SubNet(nn.Module):
    """One feature -> one number. ffn='relu': Linear(1,hid)-ReLU-Linear(hid,1).
       ffn='reglu': Linear(1,2*hid) split into (value, gate) -> value * ReLU(gate) -> Linear(hid,1)."""
    def __init__(self, ffn="relu", hid=64):
        super().__init__()
        self.ffn = ffn
        self.fc1 = nn.Linear(1, hid if ffn == "relu" else 2 * hid); self.fc2 = nn.Linear(hid, 1); self.relu = nn.ReLU()
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity="relu"); nn.init.zeros_(self.fc1.bias)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity="relu"); nn.init.zeros_(self.fc2.bias)
    def forward(self, xj):
        z = self.fc1(xj)
        if self.ffn == "reglu":
            val, gate = z.chunk(2, dim=-1); a = val * self.relu(gate)
        else:
            a = self.relu(z)
        return self.fc2(a)

class NAM(nn.Module):
    def __init__(self, n_feat=13, ffn="relu", hid=64):
        super().__init__()
        self.subs = nn.ModuleList([SubNet(ffn, hid) for _ in range(n_feat)])
        self.b0 = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        return sum(sub(x[:, j:j + 1]) for j, sub in enumerate(self.subs)) + self.b0

# ReLU baseline: hidden 64 (193 parameters per subnet). ReGLU "same hidden": 64 (321 per subnet).
# ReGLU "param-matched": hidden 38 (191 per subnet, closest to 193).
VARIANTS = {
    "ReLU (baseline)":       dict(ffn="relu", hid=64),
    "ReGLU, same hidden":    dict(ffn="reglu", hid=64),
    "ReGLU, param-matched":  dict(ffn="reglu", hid=38),
}
DELTA = 0.5   # Huber delta (log-z space), as V2_0
def make_model(v, seed):
    torch.manual_seed(seed)
    return NAM(n_feat=len(REG_FEATURES), **VARIANTS[v]).to(device)


def load_into_torch(m, Pn):
    with torch.no_grad():
        for f, sub in enumerate(m.subs):
            sub.fc1.weight.copy_(torch.tensor(Pn["W1"][f, 0, :]).view(-1, 1)); sub.fc1.bias.copy_(torch.tensor(Pn["b1"][f]))
            sub.fc2.weight.copy_(torch.tensor(Pn["W2"][f, :, 0]).view(1, -1)); sub.fc2.bias.copy_(torch.tensor(Pn["b2"][f]))
        m.b0.copy_(torch.tensor(Pn["b0"]))

for vname, r_ in last.items():
    spec, P_, xb, yb_ = r_["spec"], r_["P"], r_["Xb"][:64], r_["zb"][:64]
    FFN, HH = spec["ffn"], spec["hid"]; HW = HH if FFN == "relu" else 2 * HH
    m = make_model(vname, 0).double().eval(); load_into_torch(m, P_)
    pt_ = m(torch.tensor(xb)); loss_t = nn.HuberLoss(delta=DELTA)(pt_, torch.tensor(yb_).view(-1, 1)); loss_t.backward()
    ps, cache = fwd(xb); loss_s, dp = huber(ps, yb_)
    G = {k: np.zeros_like(a) for k, a in P_.items()}; bwd(dp, cache, G)
    d_out = float(np.abs(pt_.detach().numpy() - ps).max()); d_loss = abs(float(loss_t) - loss_s)
    g_t = {"W1": np.stack([s.fc1.weight.grad.numpy().ravel() for s in m.subs])[:, None, :], "b1": np.stack([s.fc1.bias.grad.numpy() for s in m.subs]),
           "W2": np.stack([s.fc2.weight.grad.numpy().ravel() for s in m.subs])[:, :, None], "b2": np.stack([s.fc2.bias.grad.numpy() for s in m.subs]), "b0": m.b0.grad.numpy()}
    d_grad = max(float(np.abs(g_t[k] - G[k]).max() / (np.abs(g_t[k]).max() + 1e-12)) for k in G)
    print(f"{vname:22s} | max|dpred| {d_out:.2e} | dloss {d_loss:.2e} | max rel grad diff {d_grad:.2e}")
    assert d_out < 1e-8 and d_loss < 1e-8 and d_grad < 1e-6, "scratch and PyTorch disagree on identical weights"
print("scratch forward and backward match PyTorch on identical weights (last fold)")


ReLU (baseline)        | max|dpred| 2.00e-15 | dloss 0.00e+00 | max rel grad diff 3.32e-14
ReGLU, param-matched   | max|dpred| 7.77e-15 | dloss 8.33e-17 | max rel grad diff 3.27e-14
scratch forward and backward match PyTorch on identical weights (last fold)
